# Slice-Store Optimization Benchmark Results

Reads `results/<run-id>/results.csv` produced by `benchmark.py` and compares the four
slice-store configuration corners plus the `max_slice_group_size` sweep.

Two throughput measures per cell:
- **e2e**: input tuples / query wall time (systest `-b`)
- **tl**: mean of the ThroughputListener interval samples (true parsed-source-tuple rate)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

### Categorical palette (validated, fixed order); text stays in ink colors, marks carry identity.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
INK, INK2 = "#0b0b0b", "#52514e"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "text.color": INK, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.edgecolor": "#d8d7d2", "axes.grid": True, "grid.color": "#e8e7e2",
    "axes.spines.top": False, "axes.spines.right": False,
})

### Latest run by default; set RUN_ID to pin one.
RUN_ID = None
results_root = Path("results")
runs = sorted(p for p in results_root.iterdir() if p.is_dir())
run_dir = (results_root / RUN_ID) if RUN_ID else (runs[-1] if runs else results_root)
df = pd.read_csv(run_dir / "results.csv")
df["config"] = df.apply(lambda r: {
    (False, False): "baseline",
    (True, False): "recycling",
    (False, True): "group creation",
    (True, True): "both",
}[(bool(r.enable_slice_recycling), bool(r.enable_slice_group_creation))], axis=1)
CONFIG_ORDER = ["baseline", "recycling", "group creation", "both"]
ok = df[df.failure_reason.isna() | (df.failure_reason == "")]
failed = df[~df.index.isin(ok.index)]
print(f"run {run_dir.name}: {len(ok)} ok cells, {len(failed)} failed")
failed[["query", "config", "max_slice_group_size", "threads", "failure_reason"]] if len(failed) else None

## 2×2 corners per query — e2e and ThroughputListener side by side

In [ ]:
import numpy as np

DEFAULT_SIZE = 256
DEFAULT_POOL = 0

def corner_bars(metric, title, threads):
    d = ok[(ok.threads == threads) & (ok.max_slice_group_size == DEFAULT_SIZE)
           & (ok.slice_pool_capacity == DEFAULT_POOL)]
    pv = d.pivot_table(index="query", columns="config", values=metric).reindex(columns=CONFIG_ORDER)
    pv = pv.dropna(how="all")
    x = np.arange(len(pv))
    w = 0.2
    fig, ax = plt.subplots(figsize=(max(8, len(pv) * 0.75), 4))
    for i, cfgname in enumerate(CONFIG_ORDER):
        if cfgname in pv:
            ax.bar(x + (i - 1.5) * w, pv[cfgname], width=w * 0.9, color=SERIES[i], label=cfgname)
    ax.set_xticks(x, pv.index, rotation=45, ha="right")
    ax.set_ylabel(metric)
    ax.set_title(f"{title} (threads={threads}, group size={DEFAULT_SIZE})")
    ax.legend(frameon=False)
    ax.grid(axis="x", visible=False)
    plt.tight_layout()
    plt.show()

for threads in sorted(ok.threads.unique()):
    corner_bars("e2e_tuples_per_second", "End-to-end throughput", threads)
    corner_bars("tl_mean_tps", "ThroughputListener rate", threads)

## Speedup vs baseline (e2e)

In [ ]:
for threads in sorted(ok.threads.unique()):
    d = ok[(ok.threads == threads) & (ok.max_slice_group_size == DEFAULT_SIZE)
           & (ok.slice_pool_capacity == DEFAULT_POOL)]
    pv = d.pivot_table(index="query", columns="config", values="e2e_tuples_per_second").reindex(columns=CONFIG_ORDER)
    speedup = pv.div(pv["baseline"], axis=0).drop(columns=["baseline"]).dropna(how="all")
    fig, ax = plt.subplots(figsize=(6, max(3, len(speedup) * 0.35)))
    im = ax.imshow(speedup, cmap="RdBu_r", vmin=0.5, vmax=1.5, aspect="auto")
    ax.set_xticks(range(len(speedup.columns)), speedup.columns)
    ax.set_yticks(range(len(speedup)), speedup.index)
    for (i, j), v in np.ndenumerate(speedup.values):
        if not np.isnan(v):
            ax.text(j, i, f"{v:.2f}x", ha="center", va="center", fontsize=8, color=INK)
    ax.set_title(f"Speedup vs baseline (threads={threads})")
    ax.grid(visible=False)
    fig.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.show()

## max_slice_group_size sweep (group creation on)

In [ ]:
d = ok[(ok.enable_slice_group_creation == True) & (ok.slice_pool_capacity == DEFAULT_POOL)]
for threads in sorted(d.threads.unique()):
    dt = d[d.threads == threads]
    queries = sorted(dt["query"].unique())
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for qi, q in enumerate(queries):
        for ri, recycling in enumerate([False, True]):
            dq = dt[(dt["query"] == q) & (dt.enable_slice_recycling == recycling)].sort_values("max_slice_group_size")
            if len(dq) > 1:
                ax.plot(dq.max_slice_group_size, dq.e2e_tuples_per_second,
                        marker="o", ms=5, lw=2, ls="-" if recycling else "--",
                        color=SERIES[qi % len(SERIES)],
                        label=f"{q}{' +recycling' if recycling else ''}")
    ax.set_xscale("log", base=2)
    ax.set_xlabel("max_slice_group_size")
    ax.set_ylabel("e2e tuples/s")
    ax.set_title(f"Group-size sweep (threads={threads}; dashed = recycling off)")
    ax.legend(frameon=False, fontsize=8, ncols=2)
    plt.tight_layout()
    plt.show()

## slice_pool_capacity sweep (recycling on)

The slice store reads `slice_pool_capacity` only while recycling is on, so the sweep is
pruned to those cells and there is nothing to plot for the baseline corners.
`0` = auto = `max(windowSize / windowSlide, 1)`.

In [ ]:
d = ok[(ok.enable_slice_recycling == True) & (ok.max_slice_group_size == DEFAULT_SIZE)]
for threads in sorted(d.threads.unique()):
    dt = d[d.threads == threads]
    pv = dt.pivot_table(index="query", columns=["enable_slice_group_creation", "slice_pool_capacity"],
                        values="e2e_tuples_per_second").dropna(how="all")
    if pv.empty:
        continue
    x = np.arange(len(pv))
    w = 0.8 / len(pv.columns)
    fig, ax = plt.subplots(figsize=(max(6, len(pv) * 1.6), 4))
    for i, col in enumerate(pv.columns):
        gc, pool = col
        ax.bar(x + (i - (len(pv.columns) - 1) / 2) * w, pv[col], width=w * 0.9,
               color=SERIES[i % len(SERIES)],
               label=f"pool={'auto' if pool == 0 else pool}{', grouping' if gc else ''}")
    ax.set_xticks(x, pv.index)
    ax.set_ylabel("e2e tuples/s")
    ax.set_title(f"Pool capacity, recycling on (threads={threads}, group size={DEFAULT_SIZE})")
    ax.legend(frameon=False, fontsize=8)
    ax.grid(axis="x", visible=False)
    plt.tight_layout()
    plt.show()

### A pool that never fills and a pool that is never hit look identical in e2e; the creation
### time is what separates them.
d.pivot_table(index=["query", "threads"], columns="slice_pool_capacity",
              values=["slice_creation_ms", "slices_created"])


## Slice-creation statistics vs throughput

In [ ]:
d = ok.dropna(subset=["slices_created"]) if "slices_created" in ok else ok.iloc[0:0]
if len(d):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for i, cfgname in enumerate(CONFIG_ORDER):
        dc = d[d.config == cfgname]
        axes[0].scatter(dc.slices_created, dc.e2e_tuples_per_second, s=30, color=SERIES[i], label=cfgname, alpha=0.8)
        axes[1].scatter(dc.slices_wasted, dc.e2e_tuples_per_second, s=30, color=SERIES[i], alpha=0.8)
    axes[0].set_xlabel("slices created"); axes[0].set_ylabel("e2e tuples/s")
    axes[1].set_xlabel("slices wasted (lost creation races)")
    axes[0].set_xscale("symlog"); axes[1].set_xscale("symlog")
    axes[0].legend(frameon=False)
    fig.suptitle("Slice creation work vs throughput")
    plt.tight_layout()
    plt.show()
else:
    print("no slice statistics in this run (NES_LOG_LEVEL below INFO?)")

## Full table

In [ ]:
cols = ["query", "config", "max_slice_group_size", "slice_pool_capacity", "threads", "e2e_time_s",
        "e2e_tuples_per_second", "tl_mean_tps", "tl_p95_tps", "slices_created", "slices_wasted", "failure_reason"]
df.sort_values(["query", "threads", "config"])[[c for c in cols if c in df.columns]]